# Course 8: Privacy-Preserving Machine Learning
## Session 6: Attacking and defending federated learning

**Goal:** rebuild a training image from the gradient a client shares, then add noise and watch the attack fail.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. The setting
A client computes the gradient on one private image and sends it to the server. The server knows the model. Can it recover the image?

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
torch.manual_seed(0); torch.set_default_dtype(torch.float64)
d = load_digits()
x_true = torch.tensor(d.data[7] / 16.0); y_true = torch.tensor(int(d.target[7]))
model = nn.Sequential(nn.Linear(64, 32), nn.Sigmoid(), nn.Linear(32, 10))

def shared_gradient(x, y):
    loss = F.cross_entropy(model(x[None]), y[None])
    return [g.detach() for g in torch.autograd.grad(loss, model.parameters())]

g_true = shared_gradient(x_true, y_true)
print("the client shares", sum(g.numel() for g in g_true), "numbers; the image has", x_true.numel())

### 2. Attack one: read the image straight off the first layer
For a first layer `z = W x + b`, the gradients satisfy `dL/dW[i] = dL/db[i] * x`. Dividing one by the other gives the input exactly.

In [ ]:
dW, db = g_true[0], g_true[1]
i = db.abs().argmax()
x_analytic = dW[i] / db[i]
print("largest pixel error:", (x_analytic - x_true).abs().max().item())
assert torch.allclose(x_analytic, x_true, atol=1e-8)

### 3. Attack two: gradient matching (deep leakage from gradients)
This works for more general models. Start from a random image and change it until its gradient points the same way as the shared one. The label can be read off first: with cross-entropy and one example, the output-bias gradient is negative only at the true class.

In [ ]:
y_guess = g_true[-1].argmin()
print("label recovered from the gradient:", y_guess.item(), "| true label:", y_true.item())

def invert(g_target, steps=60, seed=0):
    torch.manual_seed(seed)
    x = torch.rand(64, requires_grad=True); nt = torch.sqrt(sum((b ** 2).sum() for b in g_target))
    opt = torch.optim.LBFGS([x], lr=1.0, max_iter=20, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(model(x[None]), y_guess[None])
        g = torch.autograd.grad(loss, model.parameters(), create_graph=True)
        dot = sum((a * b).sum() for a, b in zip(g, g_target)); na = torch.sqrt(sum((a ** 2).sum() for a in g))
        diff = 1 - dot / (na * nt)                             # cosine distance: the direction matters, not the length
        diff.backward(); return diff
    for _ in range(steps): opt.step(closure)
    return x.detach()

x_rec = invert(g_true)
mse_clean = ((x_rec - x_true) ** 2).mean().item()
print("reconstruction error (mean squared): %.2e" % mse_clean)
assert mse_clean < 1e-3

### 4. The defence: clip and add noise before sharing
This is DP applied to the client's update. Watch both attacks degrade as the noise grows.

In [ ]:
import matplotlib.pyplot as plt
def defended(sigma, clip=1.0, seed=0):
    torch.manual_seed(seed)
    norm = torch.sqrt(sum((g ** 2).sum() for g in g_true))
    return [g * min(1.0, clip / norm.item()) + sigma * clip * torch.randn_like(g) for g in g_true]

fig, axes = plt.subplots(1, 5, figsize=(12, 3)); axes[0].imshow(x_true.view(8, 8), cmap="gray"); axes[0].set_title("private image")
results = []
for ax, sigma in zip(axes[1:], [0.0, 0.001, 0.01, 0.1]):
    g = defended(sigma)
    xa = g[0][g[1].abs().argmax()] / g[1][g[1].abs().argmax()]            # analytic attack
    xm = invert(g)                                                        # gradient-matching attack
    ea, em = ((xa - x_true) ** 2).mean().item(), ((xm - x_true) ** 2).mean().item()
    results.append((sigma, ea, em)); print(f"noise {sigma:6.3f}   analytic attack error {ea:10.3e}   matching attack error {em:10.3e}")
    ax.imshow(xm.view(8, 8).clamp(0, 1), cmap="gray"); ax.set_title(f"noise {sigma}")
fig.savefig("inversion.png")
assert results[-1][2] > 100 * results[1][2] and results[0][2] < 1e-6
print("for scale: a blank guess has error %.3f" % (x_true ** 2).mean().item())

Clipping alone (noise 0) does not help: it only rescales the gradient, and both attacks use its direction, not its length. Noise is what protects.

### 5. Secure aggregation in ten lines
Pairs of clients agree on random masks that cancel in the sum. The server learns the total and nothing about any single update. It does not hide what the total itself reveals, so it is combined with DP in practice.

In [ ]:
rng = np.random.default_rng(0)
updates = [rng.normal(size=5) for _ in range(4)]
masked = [u.copy() for u in updates]
for i in range(4):
    for j in range(i + 1, 4):
        m = rng.normal(size=5) * 100                       # a secret shared by clients i and j only
        masked[i] += m; masked[j] -= m
print("what the server sees from client 0:", masked[0].round(1))
print("true update of client 0           :", updates[0].round(2))
assert np.allclose(sum(masked), sum(updates))
print("sum of masked updates equals the true sum:", np.allclose(sum(masked), sum(updates)))

### TODO
1. Share the average gradient of a batch of 8 images instead of one. Does gradient matching still work? What does it recover?
2. Replace the sigmoid with ReLU. The analytic attack still works, but gradient matching gets harder. Why?
3. Poisoning: flip the labels at one of the ten hospitals from session 5 and measure the damage. Then replace the server's mean with a coordinate-wise median.